# PEN OIS Fair Value Model v1

**PEN OIS Conventions:** ACT/360, Annual (Zero Coupon), Nominal Rate Convention

**Tradable Maturities:** 3M, 6M, 9M, 12M

**Reference Rate:** BCRP TIBO Overnight

**Meeting Schedule:** Second Thursday of each month (rate effective next business day)

---

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from scipy.interpolate import interp1d
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# CORE ENGINE: PEN OIS Fair Value Calculator
# ============================================================

from scipy.optimize import minimize

def get_second_thursday(year, month):
    """Return second Thursday of a given month."""
    first_day = datetime(year, month, 1)
    day_of_week = first_day.weekday()
    days_to_thursday = (3 - day_of_week) % 7
    first_thursday = first_day + timedelta(days=days_to_thursday)
    second_thursday = first_thursday + timedelta(days=7)
    return second_thursday


def generate_meeting_dates(start_date, n_meetings=14):
    """Generate next n BCRP meeting dates from start_date.
    Meeting = 2nd Thursday of each month. Rate effective next biz day."""
    meetings = []
    year = start_date.year
    month = start_date.month
    while len(meetings) < n_meetings:
        mt = get_second_thursday(year, month)
        effective_date = mt + timedelta(days=1)
        while effective_date.weekday() >= 5:
            effective_date += timedelta(days=1)
        if mt >= start_date:
            meetings.append({
                'meeting_date': mt,
                'effective_date': effective_date,
                'label': mt.strftime('%d-%b-%y')
            })
        month += 1
        if month > 12:
            month = 1
            year += 1
    return meetings


def pen_ois_fv(value_date, maturity_date, tibo_current, meetings, path_bps):
    """Calculate PEN OIS Fair Value using piecewise constant forward rates.
    PEN OIS: ACT/360, Annual, Zero Coupon, Nominal (Simple) Rate Convention.

    Procedure:
    1. Build piecewise constant ON rate schedule from meeting date changes.
    2. Compound daily: CF = Product of (1 + r_day / 360) for each calendar day.
    3. Back out nominal (simple) rate: R = (CF - 1) * 360 / D
       where D = ACT days from value_date to maturity_date.
    """
    # Build the ON rate path: piecewise constant from meeting to meeting
    rate_changes = []
    current_rate = tibo_current
    for i, m in enumerate(meetings):
        if i < len(path_bps):
            current_rate += path_bps[i] / 100.0  # bps input -> pct change
        rate_changes.append((m['effective_date'], current_rate))

    total_days = (maturity_date - value_date).days
    if total_days <= 0:
        return tibo_current

    # Compound daily ON rates (ACT/360 simple interest per day)
    compounded = 1.0
    d = value_date
    while d < maturity_date:
        # Determine ON rate for this day (piecewise constant)
        on_rate = tibo_current  # before any meeting
        for eff_date, rate in rate_changes:
            if d >= eff_date:
                on_rate = rate
            else:
                break
        # Daily accrual: simple interest for 1 calendar day
        compounded *= (1.0 + on_rate / 100.0 / 360.0)
        d += timedelta(days=1)

    # Back out nominal (simple) annualized rate, ACT/360
    ois_rate = (compounded - 1.0) * 360.0 / total_days * 100.0
    return ois_rate


def compute_fv_curve(value_date, tenors_days, tibo_current, meetings, path_bps):
    """Compute FV for each tenor."""
    results = {}
    for label, days in tenors_days.items():
        mat_date = value_date + timedelta(days=days)
        fv = pen_ois_fv(value_date, mat_date, tibo_current, meetings, path_bps)
        results[label] = fv
    return results


def calibrate_market_implied_weights(scenario_fvs, mkt_rates, tenors, prior_weights=None):
    """Find scenario weights that best fit market OIS rates.

    Solves:  min || FV_matrix @ w - mkt_vector ||^2 + lambda * || w - w_prior ||^2
             s.t.  sum(w) = 1,  w_i >= 0

    This uses the scenario paths (which encode timing of cuts/hikes) and lets the
    market rates determine the probability mix — no flat-forward assumptions needed.

    Args:
        scenario_fvs: dict {scenario_name: {tenor: fv_rate}}
        mkt_rates: dict {tenor: market_rate}
        tenors: dict {tenor_label: days}
        prior_weights: dict {scenario_name: weight_pct} or None (uniform)

    Returns:
        dict {scenario_name: weight_pct} — market-implied weights (sum to 100)
    """
    names = list(scenario_fvs.keys())
    tenor_labels = list(tenors.keys())
    n_sc = len(names)
    n_tenors = len(tenor_labels)

    # Build FV matrix (n_tenors x n_scenarios)
    fv_matrix = np.zeros((n_tenors, n_sc))
    mkt_vec = np.zeros(n_tenors)
    for i, t in enumerate(tenor_labels):
        mkt_vec[i] = mkt_rates[t]
        for j, name in enumerate(names):
            fv_matrix[i, j] = scenario_fvs[name][t]

    # Prior weights
    if prior_weights is not None:
        w_prior = np.array([prior_weights.get(n, 100.0 / n_sc) for n in names]) / 100.0
    else:
        w_prior = np.ones(n_sc) / n_sc

    # Regularization: small pull toward prior (keeps it well-conditioned)
    reg_lambda = 0.01

    def objective(w):
        resid = fv_matrix @ w - mkt_vec
        # Weight residuals in bps^2
        fit_err = np.sum((resid * 100.0) ** 2)
        reg_err = reg_lambda * np.sum(((w - w_prior) * 100.0) ** 2)
        return fit_err + reg_err

    # Constraints: sum = 1
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    # Bounds: 0 <= w_i <= 1
    bounds = [(0.0, 1.0)] * n_sc

    result = minimize(objective, w_prior, method='SLSQP',
                      bounds=bounds, constraints=constraints,
                      options={'ftol': 1e-12, 'maxiter': 500})

    implied_weights = {}
    for j, name in enumerate(names):
        implied_weights[name] = result.x[j] * 100.0

    return implied_weights


def compute_implied_path_from_weights(tibo, meetings, scenarios, weights):
    """From probability-weighted scenarios, compute the implied ON rate at each meeting.

    Returns per-meeting results: implied ON, meeting change, cumulative change, prob of 25bps.
    """
    n_meetings = len(meetings)
    results = []

    for i in range(n_meetings):
        # Weighted ON rate after meeting i
        wt_rate = 0.0
        for name, path in scenarios.items():
            w = weights.get(name, 0) / 100.0
            r = tibo
            for j in range(i + 1):
                if j < len(path):
                    r += path[j] / 100.0
            wt_rate += w * r

        # Previous weighted rate (before this meeting, i.e., after meeting i-1)
        if i == 0:
            prev_rate = tibo
        else:
            prev_rate = results[i - 1]['implied_on']

        meeting_chg_bps = (wt_rate - prev_rate) * 100.0
        cum_chg_bps = (wt_rate - tibo) * 100.0
        prob_25 = meeting_chg_bps / 25.0 * 100.0  # % prob of a 25bps move
        cum_cuts_25 = cum_chg_bps / 25.0

        results.append({
            'meeting': meetings[i]['meeting_date'].strftime('%d-%b-%y'),
            'implied_on': wt_rate,
            'meeting_chg_bps': meeting_chg_bps,
            'cum_chg_bps': cum_chg_bps,
            'prob_25bps': prob_25,
            'cum_cuts_25': cum_cuts_25,
        })

    return results


def generate_simulated_history(current_rate, n_days=252, vol_bps=3.0):
    """Generate simulated 1Y lookback data for a rate."""
    np.random.seed(42)
    rates = np.zeros(n_days)
    rates[0] = current_rate + np.random.normal(0, 0.05)
    for i in range(1, n_days):
        drift = 0.02 * (current_rate - rates[i-1])
        shock = np.random.normal(0, vol_bps / 100.0)
        rates[i] = rates[i-1] + drift + shock
    end_date = datetime(2026, 3, 10)
    dates = pd.bdate_range(end=end_date, periods=n_days)
    return pd.Series(rates, index=dates)

In [ ]:
# ============================================================
# CONFIGURATION & STATE
# ============================================================

VALUE_DATE = datetime(2026, 3, 10)

TENORS = {
    '3M': 91,
    '6M': 182,
    '9M': 273,
    '12M': 365
}

# Current market data
TIBO_ON = 4.25  # Current TIBO ON Rate (%)

MKT_RATES = {
    '3M': 4.15,
    '6M': 4.10,
    '9M': 4.095,
    '12M': 4.095
}

# Generate meeting dates
MEETINGS = generate_meeting_dates(VALUE_DATE, n_meetings=14)

# Default scenario paths (bps change per meeting)
DEFAULT_SCENARIOS = {
    'Hold':          [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'Gradual Cut':   [0, -25, 0, -25, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'Aggr Cut':      [-25, -25, -25, -25, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'Front Cut':     [-25, -25, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'Back Cut':      [0, 0, 0, 0, 0, -25, -25, 0, 0, 0, 0, 0, 0, 0],
    'Hike':          [0, 25, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'Deep Cut':      [-25, -25, -25, -25, -25, -25, 0, 0, 0, 0, 0, 0, 0, 0],
}

# Default weights (must sum to 100)
DEFAULT_WEIGHTS = {
    'Hold':        5.0,
    'Gradual Cut': 35.0,
    'Aggr Cut':    10.0,
    'Front Cut':   25.0,
    'Back Cut':    10.0,
    'Hike':        5.0,
    'Deep Cut':    10.0,
}

print(f"Value Date: {VALUE_DATE.strftime('%d-%b-%Y')}")
print(f"TIBO ON: {TIBO_ON}%")
print(f"Market Rates: {MKT_RATES}")
print(f"\nBCRP Meeting Dates:")
for i, m in enumerate(MEETINGS):
    print(f"  {i+1}. {m['meeting_date'].strftime('%d-%b-%Y')} (Thu) -> Effective: {m['effective_date'].strftime('%d-%b-%Y')}")

In [ ]:
# ============================================================
# SECTION 1: INPUT PATHS TABLE (Interactive)
# ============================================================

# --- CSS Styling ---
DARK_CSS = """
<style>
.pen-container { background: #0a1628; color: #c8d6e5; font-family: 'Segoe UI', Consolas, monospace; padding: 15px; border-radius: 8px; margin: 5px 0; }
.pen-title { color: #00e6c3; font-weight: bold; font-size: 14px; margin-bottom: 10px; letter-spacing: 1px; }
.pen-table { border-collapse: collapse; width: 100%; font-size: 12px; }
.pen-table th { background: #162744; color: #7fbbf0; padding: 6px 10px; text-align: center; border: 1px solid #1e3a5f; font-size: 11px; }
.pen-table td { padding: 5px 8px; text-align: center; border: 1px solid #1e3a5f; }
.pen-table tr:nth-child(even) { background: #0d1f3c; }
.pen-table tr:nth-child(odd) { background: #0a1628; }
.val-pos { color: #00e676; font-weight: bold; }
.val-neg { color: #ff5252; font-weight: bold; }
.val-zero { color: #78909c; }
.val-fv { color: #ffd740; font-weight: bold; }
.row-mkt { background: #1a2744 !important; }
.row-fv { background: #1b3025 !important; }
.row-delta { background: #2a1a3e !important; }
.row-high { background: #0d3333 !important; }
.row-low { background: #33150d !important; }
.header-scenario { text-align: left !important; }
.weight-cell { color: #82b1ff; }
</style>
"""


def format_bps(val):
    if val > 0:
        return f'<span class="val-pos">+{val:.0f}</span>'
    elif val < 0:
        return f'<span class="val-neg">{val:.0f}</span>'
    return '<span class="val-zero">0</span>'


def format_rate(val, css_class=''):
    cls = f' class="{css_class}"' if css_class else ''
    return f'<span{cls}>{val:.3f}</span>'


def format_delta_bps(val):
    if val > 0:
        return f'<span class="val-pos">{val:.1f}</span>'
    elif val < 0:
        return f'<span class="val-neg">{val:.1f}</span>'
    return f'<span class="val-zero">{val:.1f}</span>'


def render_paths_table(scenarios, weights, meetings):
    """Render the input paths table similar to the screenshot."""
    meeting_labels = [m['meeting_date'].strftime('%d-%b-%y') for m in meetings]
    
    html = DARK_CSS + '<div class="pen-container">'
    html += '<div class="pen-title">INPUT PATHS (MAX 14 MEETINGS)</div>'
    html += '<table class="pen-table">'
    
    # Header
    html += '<tr><th>Weight %</th><th class="header-scenario">Scenario</th>'
    for lbl in meeting_labels:
        html += f'<th>{lbl}</th>'
    html += '</tr>'
    
    # Rows per scenario
    for name, path in scenarios.items():
        w = weights.get(name, 0)
        html += f'<tr><td class="weight-cell">{w:.0f}</td>'
        html += f'<td class="header-scenario" style="color:#4fc3f7">{name}</td>'
        for i, lbl in enumerate(meeting_labels):
            val = path[i] if i < len(path) else 0
            html += f'<td>{format_bps(val)}</td>'
        html += '</tr>'
    
    # Total weight row
    total_w = sum(weights.values())
    html += f'<tr style="background:#162744"><td style="color:#ffd740;font-weight:bold">{total_w:.0f}</td>'
    html += '<td class="header-scenario" style="color:#ffd740;font-weight:bold">TOTAL</td>'
    for _ in meeting_labels:
        html += '<td></td>'
    html += '</tr>'
    
    html += '</table></div>'
    return html


display(HTML(render_paths_table(DEFAULT_SCENARIOS, DEFAULT_WEIGHTS, MEETINGS)))

In [ ]:
# ============================================================
# SECTION 2: MARKET DATA & TIBO INPUT
# ============================================================

def render_market_input(tibo, mkt_rates):
    html = DARK_CSS + '<div class="pen-container">'
    html += '<div class="pen-title">MARKET DATA INPUT</div>'
    html += '<table class="pen-table">'
    html += '<tr><th>Parameter</th><th>Value (%)</th></tr>'
    html += f'<tr><td style="color:#ffd740;font-weight:bold">TIBO ON Rate</td><td style="color:#ffd740;font-weight:bold">{tibo:.3f}</td></tr>'
    for tenor, rate in mkt_rates.items():
        html += f'<tr><td>PEN OIS {tenor}</td><td>{rate:.3f}</td></tr>'
    html += '</table></div>'
    return html

display(HTML(render_market_input(TIBO_ON, MKT_RATES)))

In [ ]:
# ============================================================
# SECTION 3: FV TABLE WITH DELTAS
# ============================================================

def compute_all_fv(value_date, tenors, tibo, meetings, scenarios, weights):
    """Compute FV for all scenarios and the probability-weighted FV."""
    scenario_fvs = {}
    for name, path in scenarios.items():
        fvs = compute_fv_curve(value_date, tenors, tibo, meetings, path)
        scenario_fvs[name] = fvs
    
    # Probability-weighted FV
    weighted_fv = {}
    for tenor in tenors:
        wt_sum = 0.0
        for name, fvs in scenario_fvs.items():
            w = weights.get(name, 0) / 100.0
            wt_sum += w * fvs[tenor]
        weighted_fv[tenor] = wt_sum
    
    # Find highest and lowest FV paths
    high_fv = {}
    low_fv = {}
    high_name = {}
    low_name = {}
    for tenor in tenors:
        vals = {name: fvs[tenor] for name, fvs in scenario_fvs.items()}
        max_name = max(vals, key=vals.get)
        min_name = min(vals, key=vals.get)
        high_fv[tenor] = vals[max_name]
        low_fv[tenor] = vals[min_name]
        high_name[tenor] = max_name
        low_name[tenor] = min_name
    
    return scenario_fvs, weighted_fv, high_fv, low_fv, high_name, low_name


def render_fv_table(tenors, mkt_rates, scenario_fvs, weighted_fv, high_fv, low_fv, weights):
    """Render the FV table with scenario rates and deltas."""
    tenor_labels = list(tenors.keys())
    
    html = DARK_CSS + '<div class="pen-container">'
    html += '<div class="pen-title">FAIR VALUE TABLE — PEN OIS (ACT/360 Ann ZC)</div>'
    html += '<table class="pen-table">'
    
    # Header
    html += '<tr><th>Weight %</th><th class="header-scenario">Scenario</th>'
    for t in tenor_labels:
        html += f'<th>PEN OIS {t}</th>'
    html += '</tr>'
    
    # Scenario rows
    for name, fvs in scenario_fvs.items():
        w = weights.get(name, 0)
        html += f'<tr><td class="weight-cell">{w:.1f}</td>'
        html += f'<td class="header-scenario" style="color:#4fc3f7">{name}</td>'
        for t in tenor_labels:
            # Color: closer to mkt = lighter
            val = fvs[t]
            diff = (val - mkt_rates[t]) * 100  # bps
            if diff > 2:
                bg = '#1a3322'
            elif diff < -2:
                bg = '#331a1a'
            else:
                bg = ''
            style = f' style="background:{bg}"' if bg else ''
            html += f'<td{style}>{val:.3f}</td>'
        html += '</tr>'
    
    # Market row
    html += '<tr class="row-mkt"><td></td><td class="header-scenario" style="color:#ff9800;font-weight:bold">Mkt</td>'
    for t in tenor_labels:
        html += f'<td style="color:#ff9800;font-weight:bold">{mkt_rates[t]:.3f}</td>'
    html += '</tr>'
    
    # Weighted FV row
    html += '<tr class="row-fv"><td style="color:#ffd740;font-weight:bold">100</td>'
    html += '<td class="header-scenario" style="color:#ffd740;font-weight:bold">FV (Wtd)</td>'
    for t in tenor_labels:
        html += f'<td style="color:#ffd740;font-weight:bold">{weighted_fv[t]:.3f}</td>'
    html += '</tr>'
    
    # Mkt - FV delta (bps)
    html += '<tr class="row-delta"><td></td><td class="header-scenario" style="color:#ce93d8;font-weight:bold">Mkt − FV (bps)</td>'
    for t in tenor_labels:
        delta = (mkt_rates[t] - weighted_fv[t]) * 100
        html += f'<td>{format_delta_bps(delta)}</td>'
    html += '</tr>'
    
    # Weighted EV (bps) = abs(Mkt - FV)
    html += '<tr class="row-delta"><td></td><td class="header-scenario" style="color:#ce93d8">Weighted EV (bps)</td>'
    for t in tenor_labels:
        ev = abs(mkt_rates[t] - weighted_fv[t]) * 100
        html += f'<td>{format_delta_bps(ev)}</td>'
    html += '</tr>'
    
    # High Delta Active (bps): Mkt - highest scenario FV
    html += '<tr class="row-high"><td></td><td class="header-scenario" style="color:#26a69a;font-weight:bold">High Δ Active (bps)</td>'
    for t in tenor_labels:
        delta = (high_fv[t] - mkt_rates[t]) * 100
        html += f'<td>{format_delta_bps(delta)}</td>'
    html += '</tr>'
    
    # Low Delta Active (bps): Mkt - lowest scenario FV
    html += '<tr class="row-low"><td></td><td class="header-scenario" style="color:#ef5350;font-weight:bold">Low Δ Active (bps)</td>'
    for t in tenor_labels:
        delta = (low_fv[t] - mkt_rates[t]) * 100
        html += f'<td>{format_delta_bps(delta)}</td>'
    html += '</tr>'
    
    html += '</table></div>'
    return html


# Compute and display
scenario_fvs, weighted_fv, high_fv, low_fv, high_name, low_name = compute_all_fv(
    VALUE_DATE, TENORS, TIBO_ON, MEETINGS, DEFAULT_SCENARIOS, DEFAULT_WEIGHTS
)

display(HTML(render_fv_table(
    TENORS, MKT_RATES, scenario_fvs, weighted_fv, high_fv, low_fv, DEFAULT_WEIGHTS
)))

In [ ]:
# ============================================================
# SECTION 4: MARKET-IMPLIED PROBABILITIES (Scenario Calibration)
# ============================================================
# 
# Approach: Instead of crude interpolation, we calibrate scenario weights
# to market OIS rates. The scenarios encode the TIMING of cuts/hikes,
# and the optimizer finds the probability mix that best fits the market curve.
# From the calibrated weights, we derive per-meeting implied ON rates & probs.
#
# This avoids flat-forward assumptions — the path timing drives everything.

# Step 1: Calibrate market-implied weights
mkt_implied_weights = calibrate_market_implied_weights(
    scenario_fvs, MKT_RATES, TENORS, prior_weights=DEFAULT_WEIGHTS
)

# Step 2: Compute implied ON path from calibrated weights
mkt_implied_path = compute_implied_path_from_weights(
    TIBO_ON, MEETINGS, DEFAULT_SCENARIOS, mkt_implied_weights
)

# Step 3: Also compute implied path from USER weights (for comparison)
user_implied_path = compute_implied_path_from_weights(
    TIBO_ON, MEETINGS, DEFAULT_SCENARIOS, DEFAULT_WEIGHTS
)


def render_implied_weights_table(user_weights, mkt_weights, scenario_fvs, tenors, mkt_rates):
    """Show user weights vs market-implied weights side by side."""
    names = list(scenario_fvs.keys())
    tenor_labels = list(tenors.keys())

    html = DARK_CSS + '<div class="pen-container">'
    html += '<div class="pen-title">SCENARIO WEIGHTS — USER vs MARKET-IMPLIED</div>'
    html += '<table class="pen-table">'
    html += '<tr><th>Scenario</th><th>User Wt (%)</th><th>Mkt-Impl Wt (%)</th><th>Δ Wt</th>'
    for t in tenor_labels:
        html += f'<th>FV {t}</th>'
    html += '</tr>'

    for name in names:
        uw = user_weights.get(name, 0)
        mw = mkt_weights.get(name, 0)
        dw = mw - uw
        html += '<tr>'
        html += f'<td style="color:#4fc3f7">{name}</td>'
        html += f'<td class="weight-cell">{uw:.1f}</td>'
        html += f'<td style="color:#ffd740">{mw:.1f}</td>'
        html += f'<td>{format_delta_bps(dw)}</td>'
        for t in tenor_labels:
            html += f'<td>{scenario_fvs[name][t]:.3f}</td>'
        html += '</tr>'

    # Totals
    html += '<tr style="background:#162744">'
    html += '<td style="color:#ffd740;font-weight:bold">TOTAL</td>'
    html += f'<td style="color:#ffd740">{sum(user_weights.values()):.1f}</td>'
    html += f'<td style="color:#ffd740">{sum(mkt_weights.values()):.1f}</td>'
    html += '<td></td>'
    # Mkt-implied weighted FV
    for t in tenor_labels:
        wfv = sum(mkt_weights.get(n, 0) / 100.0 * scenario_fvs[n][t] for n in names)
        html += f'<td style="color:#ffd740;font-weight:bold">{wfv:.3f}</td>'
    html += '</tr>'

    # Market rates for comparison
    html += '<tr style="background:#1a2744">'
    html += '<td style="color:#ff9800;font-weight:bold">Market</td><td></td><td></td><td></td>'
    for t in tenor_labels:
        html += f'<td style="color:#ff9800;font-weight:bold">{mkt_rates[t]:.3f}</td>'
    html += '</tr>'

    # Fit error
    html += '<tr style="background:#2a1a3e">'
    html += '<td style="color:#ce93d8">Fit Error (bps)</td><td></td><td></td><td></td>'
    for t in tenor_labels:
        wfv = sum(mkt_weights.get(n, 0) / 100.0 * scenario_fvs[n][t] for n in names)
        err = (wfv - mkt_rates[t]) * 100
        html += f'<td>{format_delta_bps(err)}</td>'
    html += '</tr>'

    html += '</table></div>'
    return html


def render_prob_table(probs, title_suffix=''):
    html = DARK_CSS + '<div class="pen-container">'
    html += f'<div class="pen-title">IMPLIED PROBABILITIES — BCRP MEETINGS (25bps){title_suffix}</div>'
    html += '<table class="pen-table">'
    html += '<tr><th>Meeting</th><th>Impl ON (%)</th><th>Mtg Δ (bps)</th><th>Cum Δ (bps)</th>'
    html += '<th>Prob 25bps (%)</th><th>Cum Cuts (25bps)</th></tr>'

    for p in probs:
        prob_color = 'val-neg' if p['prob_25bps'] < 0 else ('val-pos' if p['prob_25bps'] > 0 else 'val-zero')
        cum_color = 'val-neg' if p['cum_chg_bps'] < 0 else ('val-pos' if p['cum_chg_bps'] > 0 else 'val-zero')

        html += '<tr>'
        html += f'<td style="color:#4fc3f7">{p["meeting"]}</td>'
        html += f'<td>{p["implied_on"]:.3f}</td>'
        html += f'<td>{format_delta_bps(p["meeting_chg_bps"])}</td>'
        html += f'<td class="{cum_color}">{p["cum_chg_bps"]:.1f}</td>'
        html += f'<td class="{prob_color}">{p["prob_25bps"]:.1f}</td>'
        html += f'<td class="{cum_color}">{p["cum_cuts_25"]:.2f}</td>'
        html += '</tr>'

    html += '</table></div>'
    return html


# Display
display(HTML(render_implied_weights_table(
    DEFAULT_WEIGHTS, mkt_implied_weights, scenario_fvs, TENORS, MKT_RATES
)))
display(HTML(render_prob_table(mkt_implied_path, ' — MARKET-IMPLIED')))
display(HTML(render_prob_table(user_implied_path, ' — USER WEIGHTS')))

In [ ]:
# ============================================================
# SECTION 5: DISTRIBUTION METRICS TABLE
# ============================================================

def compute_distribution_metrics(tenors, mkt_rates, scenario_fvs, weighted_fv, weights):
    """Compute distribution metrics for each tenor."""
    metrics = {}
    for t in tenors:
        mkt = mkt_rates[t]
        fv = weighted_fv[t]
        
        # Scenario dispersion (weighted std)
        sc_vals = []
        sc_weights = []
        for name, fvs in scenario_fvs.items():
            w = weights.get(name, 0) / 100.0
            if w > 0:
                sc_vals.append(fvs[t])
                sc_weights.append(w)
        
        sc_vals = np.array(sc_vals)
        sc_weights = np.array(sc_weights)
        
        # Weighted dispersion (bps)
        if len(sc_vals) > 1:
            wt_mean = np.average(sc_vals, weights=sc_weights)
            wt_var = np.average((sc_vals - wt_mean)**2, weights=sc_weights)
            dispersion = np.sqrt(wt_var) * 100  # bps
        else:
            dispersion = 0
        
        # Score: EV / Dispersion
        ev_bps = abs(mkt - fv) * 100
        score = ev_bps / dispersion if dispersion > 0 else 0
        
        metrics[t] = {
            'mkt_minus_fv_bps': (mkt - fv) * 100,
            'ev_bps': ev_bps,
            'dispersion_bps': dispersion,
            'score': score,
        }
    
    return metrics


def render_metrics_table(tenors, metrics):
    tenor_labels = list(tenors.keys())
    
    html = DARK_CSS + '<div class="pen-container">'
    html += '<div class="pen-title">DISTRIBUTION METRICS</div>'
    html += '<table class="pen-table">'
    
    # Header
    html += '<tr><th>Metric</th>'
    for t in tenor_labels:
        html += f'<th>PEN OIS {t}</th>'
    html += '</tr>'
    
    # Mkt - FV
    html += '<tr><td style="color:#ce93d8">Mkt − FV (bps)</td>'
    for t in tenor_labels:
        html += f'<td>{format_delta_bps(metrics[t]["mkt_minus_fv_bps"])}</td>'
    html += '</tr>'
    
    # Weighted EV
    html += '<tr><td style="color:#4fc3f7">Weighted EV (bps)</td>'
    for t in tenor_labels:
        v = metrics[t]['ev_bps']
        html += f'<td>{format_delta_bps(v)}</td>'
    html += '</tr>'
    
    # Dispersion
    html += '<tr><td style="color:#78909c">Sc Dispersion (bps)</td>'
    for t in tenor_labels:
        html += f'<td>{metrics[t]["dispersion_bps"]:.3f}</td>'
    html += '</tr>'
    
    # Score
    html += '<tr><td style="color:#ffd740;font-weight:bold">Score</td>'
    for t in tenor_labels:
        s = metrics[t]['score']
        color = '#00e676' if s > 0.7 else ('#ffd740' if s > 0.3 else '#ff5252')
        html += f'<td style="color:{color};font-weight:bold">{s:.3f}</td>'
    html += '</tr>'
    
    html += '</table></div>'
    return html


dist_metrics = compute_distribution_metrics(TENORS, MKT_RATES, scenario_fvs, weighted_fv, DEFAULT_WEIGHTS)
display(HTML(render_metrics_table(TENORS, dist_metrics)))

In [ ]:
# ============================================================
# SECTION 6: INTERACTIVE CHART — Select Tenor, See Path vs FV
# ============================================================

# Generate simulated 1Y history for each tenor
sim_history = {}
for tenor, rate in MKT_RATES.items():
    np.random.seed(hash(tenor) % 2**31)
    sim_history[tenor] = generate_simulated_history(rate, n_days=252, vol_bps=2.5)


def build_chart(tenor_key):
    """Build interactive Plotly chart for a given tenor."""
    hist = sim_history[tenor_key]
    mkt = MKT_RATES[tenor_key]
    fv = weighted_fv[tenor_key]
    hi = high_fv[tenor_key]
    lo = low_fv[tenor_key]
    
    fig = go.Figure()
    
    # Historical price series
    fig.add_trace(go.Candlestick(
        x=hist.index,
        open=hist.values - np.random.uniform(0, 0.01, len(hist)),
        high=hist.values + np.random.uniform(0, 0.02, len(hist)),
        low=hist.values - np.random.uniform(0, 0.02, len(hist)),
        close=hist.values,
        name=f'PEN OIS {tenor_key}',
        increasing_line_color='#00e676',
        decreasing_line_color='#ff5252',
    ))
    
    # FV line
    fig.add_hline(y=fv, line_dash='dash', line_color='#ffd740', line_width=1.5,
                  annotation_text=f'FV: {fv:.3f}', annotation_position='right',
                  annotation_font_color='#ffd740', annotation_font_size=11)
    
    # Market line
    fig.add_hline(y=mkt, line_dash='dot', line_color='#ff9800', line_width=1,
                  annotation_text=f'Mkt: {mkt:.3f}', annotation_position='right',
                  annotation_font_color='#ff9800', annotation_font_size=11)
    
    # Scenario level lines
    for name, fvs in scenario_fvs.items():
        w = DEFAULT_WEIGHTS.get(name, 0)
        if w > 0:
            fig.add_hline(y=fvs[tenor_key], line_dash='dot', line_color='#4fc3f7',
                          line_width=0.7, opacity=0.5,
                          annotation_text=f'{name}: {fvs[tenor_key]:.3f}',
                          annotation_position='left',
                          annotation_font_color='#4fc3f7', annotation_font_size=9)
    
    # High scenario
    fig.add_hline(y=hi, line_dash='dashdot', line_color='#26a69a', line_width=1,
                  annotation_text=f'High: {hi:.3f} ({high_name[tenor_key]})',
                  annotation_position='right',
                  annotation_font_color='#26a69a', annotation_font_size=10)
    
    # Low scenario
    fig.add_hline(y=lo, line_dash='dashdot', line_color='#ef5350', line_width=1,
                  annotation_text=f'Low: {lo:.3f} ({low_name[tenor_key]})',
                  annotation_position='right',
                  annotation_font_color='#ef5350', annotation_font_size=10)
    
    delta_bps = (mkt - fv) * 100
    delta_str = f'+{delta_bps:.1f}' if delta_bps >= 0 else f'{delta_bps:.1f}'
    
    fig.update_layout(
        title=dict(
            text=f'PEN OIS {tenor_key} — Levels Overlay (Paths, FV, Market) | Δ = {delta_str} bps | 1Y Window',
            font=dict(color='#4fc3f7', size=14),
            x=0.5
        ),
        template='plotly_dark',
        paper_bgcolor='#0a1628',
        plot_bgcolor='#0d1f3c',
        xaxis=dict(gridcolor='#1e3a5f', showgrid=True, rangeslider_visible=False),
        yaxis=dict(gridcolor='#1e3a5f', showgrid=True, title='Rate (%)',
                   tickformat='.3f'),
        height=550,
        margin=dict(r=200),
        showlegend=False,
        font=dict(family='Consolas, monospace', size=11, color='#c8d6e5'),
    )
    
    return fig


# Interactive tenor selector
tenor_dropdown = widgets.Dropdown(
    options=list(TENORS.keys()),
    value='3M',
    description='Product:',
    style={'description_width': 'auto'},
    layout=widgets.Layout(width='220px')
)

chart_output = widgets.Output()

def on_tenor_change(change):
    with chart_output:
        clear_output(wait=True)
        fig = build_chart(change['new'])
        fig.show()

tenor_dropdown.observe(on_tenor_change, names='value')

# Initial render
with chart_output:
    fig = build_chart('3M')
    fig.show()

display(widgets.VBox([tenor_dropdown, chart_output]))

In [ ]:
# ============================================================
# SECTION 7: ON RATE PATH VISUALIZATION
# ============================================================

def build_on_path_chart(tibo, meetings, scenarios, weights):
    """Visualize all ON rate paths and the weighted path."""
    fig = go.Figure()
    
    dates = [VALUE_DATE] + [m['effective_date'] for m in meetings]
    
    colors = ['#4fc3f7', '#00e676', '#ff5252', '#ffd740', '#ce93d8', '#ff9800', '#26a69a']
    
    for idx, (name, path) in enumerate(scenarios.items()):
        w = weights.get(name, 0)
        rates = [tibo]
        r = tibo
        for i in range(len(meetings)):
            if i < len(path):
                r += path[i] / 100.0
            rates.append(r)
        
        opacity = max(0.3, w / 50.0)
        color = colors[idx % len(colors)]
        
        fig.add_trace(go.Scatter(
            x=dates, y=rates,
            mode='lines+markers',
            name=f'{name} ({w:.0f}%)',
            line=dict(color=color, width=1.5, shape='hv'),
            marker=dict(size=5),
            opacity=opacity,
        ))
    
    # Weighted ON path
    weighted_rates = [tibo]
    for i in range(len(meetings)):
        wt_rate = 0
        for name, path in scenarios.items():
            w = weights.get(name, 0) / 100.0
            r = tibo
            for j in range(i + 1):
                if j < len(path):
                    r += path[j] / 100.0
            wt_rate += w * r
        weighted_rates.append(wt_rate)
    
    fig.add_trace(go.Scatter(
        x=dates, y=weighted_rates,
        mode='lines+markers',
        name='Weighted Path',
        line=dict(color='#ffd740', width=3, shape='hv'),
        marker=dict(size=7, symbol='diamond'),
    ))
    
    # Current TIBO line
    fig.add_hline(y=tibo, line_dash='dot', line_color='#78909c', line_width=1,
                  annotation_text=f'Current TIBO: {tibo:.2f}%',
                  annotation_font_color='#78909c')
    
    fig.update_layout(
        title=dict(
            text='BCRP Reference Rate Paths — Piecewise Constant ON Rate Scenarios',
            font=dict(color='#4fc3f7', size=14),
            x=0.5
        ),
        template='plotly_dark',
        paper_bgcolor='#0a1628',
        plot_bgcolor='#0d1f3c',
        xaxis=dict(gridcolor='#1e3a5f', showgrid=True, title='Date'),
        yaxis=dict(gridcolor='#1e3a5f', showgrid=True, title='ON Rate (%)',
                   tickformat='.2f'),
        height=500,
        legend=dict(x=1.02, y=1, font=dict(size=10)),
        margin=dict(r=200),
        font=dict(family='Consolas, monospace', size=11, color='#c8d6e5'),
    )
    
    return fig


fig_paths = build_on_path_chart(TIBO_ON, MEETINGS, DEFAULT_SCENARIOS, DEFAULT_WEIGHTS)
fig_paths.show()

In [ ]:
# ============================================================
# SECTION 8: FULL INTERACTIVE DASHBOARD
# ============================================================

tibo_input = widgets.FloatText(value=TIBO_ON, description='TIBO ON (%):', step=0.01,
                                style={'description_width': 'auto'},
                                layout=widgets.Layout(width='200px'))

mkt_inputs = {}
for t, r in MKT_RATES.items():
    mkt_inputs[t] = widgets.FloatText(value=r, description=f'OIS {t} (%):', step=0.005,
                                       style={'description_width': 'auto'},
                                       layout=widgets.Layout(width='200px'))

weight_inputs = {}
for name, w in DEFAULT_WEIGHTS.items():
    weight_inputs[name] = widgets.FloatText(value=w, description=f'{name}:', step=1,
                                             style={'description_width': 'auto'},
                                             layout=widgets.Layout(width='200px'))

scenario_grids = {}
for name, path in DEFAULT_SCENARIOS.items():
    row = []
    for i in range(14):
        val = path[i] if i < len(path) else 0
        w = widgets.IntText(value=val, layout=widgets.Layout(width='50px'))
        row.append(w)
    scenario_grids[name] = row

calc_button = widgets.Button(description='Recalculate FV', button_style='success',
                              layout=widgets.Layout(width='200px', height='40px'))

dash_output = widgets.Output()


def recalculate(_):
    with dash_output:
        clear_output(wait=True)

        tibo = tibo_input.value
        mkt = {t: w.value for t, w in mkt_inputs.items()}
        wts = {name: w.value for name, w in weight_inputs.items()}
        scens = {}
        for name, grid in scenario_grids.items():
            scens[name] = [g.value for g in grid]

        meetings = generate_meeting_dates(VALUE_DATE, n_meetings=14)
        sc_fvs, wt_fv, hi_fv, lo_fv, hi_nm, lo_nm = compute_all_fv(
            VALUE_DATE, TENORS, tibo, meetings, scens, wts
        )

        # Calibrate market-implied weights
        mkt_wts = calibrate_market_implied_weights(sc_fvs, mkt, TENORS, prior_weights=wts)
        mkt_path = compute_implied_path_from_weights(tibo, meetings, scens, mkt_wts)
        user_path = compute_implied_path_from_weights(tibo, meetings, scens, wts)

        display(HTML(render_paths_table(scens, wts, meetings)))
        display(HTML(render_market_input(tibo, mkt)))
        display(HTML(render_fv_table(TENORS, mkt, sc_fvs, wt_fv, hi_fv, lo_fv, wts)))
        display(HTML(render_implied_weights_table(wts, mkt_wts, sc_fvs, TENORS, mkt)))
        display(HTML(render_prob_table(mkt_path, ' — MARKET-IMPLIED')))
        display(HTML(render_prob_table(user_path, ' — USER WEIGHTS')))

        metrics = compute_distribution_metrics(TENORS, mkt, sc_fvs, wt_fv, wts)
        display(HTML(render_metrics_table(TENORS, metrics)))

        fig_p = build_on_path_chart(tibo, meetings, scens, wts)
        fig_p.show()


calc_button.on_click(recalculate)

mkt_box = widgets.VBox([tibo_input] + list(mkt_inputs.values()),
                        layout=widgets.Layout(padding='5px'))
wt_box = widgets.VBox(list(weight_inputs.values()),
                       layout=widgets.Layout(padding='5px'))

scenario_rows = []
meeting_headers = [widgets.HTML(f'<b style="color:#7fbbf0;font-size:10px;width:50px">{m["meeting_date"].strftime("%b")}</b>')
                   for m in MEETINGS]
scenario_rows.append(widgets.HBox(
    [widgets.HTML('<b style="color:#4fc3f7;width:120px">Scenario</b>')] + meeting_headers,
    layout=widgets.Layout(overflow_x='auto')
))
for name, grid in scenario_grids.items():
    row = widgets.HBox(
        [widgets.HTML(f'<span style="color:#4fc3f7;width:120px;display:inline-block">{name}</span>')] + grid,
        layout=widgets.Layout(overflow_x='auto')
    )
    scenario_rows.append(row)

sc_box = widgets.VBox(scenario_rows, layout=widgets.Layout(padding='5px', overflow_x='auto'))

controls = widgets.VBox([
    widgets.HTML('<h3 style="color:#00e6c3">Market & Weights</h3>'),
    widgets.HBox([mkt_box, wt_box]),
    widgets.HTML('<h3 style="color:#00e6c3">Scenario Paths (bps per meeting)</h3>'),
    sc_box,
    calc_button,
])

display(controls)
display(dash_output)

recalculate(None)